In [2]:
import cv2
import numpy as np

# Basic Functions

## Entropy Calculation

In [3]:
def my_entropy(input_image):

    arr = input_image.flatten().astype(np.int64)

    if arr.min() != 1:
        arr = arr - arr.min() + 1

    p = np.zeros(arr.max(), dtype=np.float64)
    for v in arr:
        p[v - 1] += 1

    p = p / p.sum()
    p = p[p != 0]
    entropy = np.sum(-p * np.log2(p))

    return entropy

## PSNR Calculation

In [4]:
def next_power_of_two(x):
    return 1 << (x - 1).bit_length()

def peak_signal_noise_ratio(image1: np.ndarray, image2: np.ndarray):
    if image1.shape != image2.shape:
        return -1

    max_intensity = max(image1.max(), image2.max())

    next_pow2 = next_power_of_two(int(max_intensity + 1))
    max_value = next_pow2 - 1
    max_value_sq = max_value ** 2

    mse = np.mean((image1.astype(np.float64) - image2.astype(np.float64)) ** 2)

    if mse == 0:
        return np.inf

    return 10 * np.log10(max_value_sq / mse)

# Stereo Compression

## Helper functions

## Image Padding

In [11]:
def pad_image(img, block_size=16):
    H, W = img.shape
    pad_h = (block_size - H % block_size) % block_size
    pad_w = (block_size - W % block_size) % block_size
    return np.pad(img, ((0, pad_h), (0, pad_w)), mode='edge'), H, W

## 3 Step Search (3SS)

In [12]:
def three_step_search(left, curr_block, y, x, block_size, search_range):
    H, W = left.shape

    step = 2 ** int(np.floor(np.log2(search_range)))
    best_mv = (0, 0)

    while step >= 1:
        best_sad = np.inf
        cy, cx = best_mv

        for dy in [-step, 0, step]:
            for dx in [-step, 0, step]:
                ny = cy + dy
                nx = cx + dx

                ref_y = y + ny
                ref_x = x + nx

                if (ref_y < 0 or ref_x < 0 or
                        ref_y + block_size > H or
                        ref_x + block_size > W):
                    continue

                ref_block = left[ref_y:ref_y+block_size,
                            ref_x:ref_x+block_size]

                sad = np.sum(np.abs(
                    curr_block.astype(np.int16) -
                    ref_block.astype(np.int16)
                ))

                if sad < best_sad:
                    best_sad = sad
                    best_mv = (ny, nx)

        step //= 2

    return best_mv

## Encoder

In [13]:
def encoder(left, right, block_size=16, search_range=22):
    left, H, W = pad_image(left, block_size)
    right, _, _ = pad_image(right, block_size)

    H_pad, W_pad = right.shape
    residual = np.zeros_like(right, dtype=np.int16)

    mv_h = H_pad // block_size
    mv_w = W_pad // block_size
    motion_vectors = np.zeros((mv_h, mv_w, 2), dtype=np.int16)

    for by in range(mv_h):
        for bx in range(mv_w):
            y = by * block_size
            x = bx * block_size

            curr_block = right[y:y+block_size, x:x+block_size]

            mv = three_step_search(left, curr_block, y, x, block_size, search_range)
            motion_vectors[by, bx] = mv

            ref_y = y + mv[0]
            ref_x = x + mv[1]

            pred_block = left[ref_y:ref_y+block_size, ref_x:ref_x+block_size]

            residual[y:y+block_size, x:x+block_size] = (
                    curr_block.astype(np.int16) -
                    pred_block.astype(np.int16))

    return motion_vectors, residual, (H, W)

## Decoder

In [14]:
def decoder(left: np.ndarray, motion_vectors, residual, orig_shape:tuple, block_size=16):
    left, _, _ = pad_image(left, block_size)

    H_pad, W_pad = residual.shape
    recon = np.zeros((H_pad, W_pad), dtype=np.int16)

    mv_h, mv_w, _ = motion_vectors.shape

    for by in range(mv_h):
        for bx in range(mv_w):
            y = by * block_size
            x = bx * block_size

            dy, dx = motion_vectors[by, bx]

            ref_y = y + dy
            ref_x = x + dx

            pred_block = left[ref_y:ref_y+block_size,
                         ref_x:ref_x+block_size]

            recon[y:y+block_size, x:x+block_size] = (
                    pred_block.astype(np.int16) +
                    residual[y:y+block_size, x:x+block_size]
            )

    H, W = orig_shape
    return recon[:H, :W].astype(np.uint8)

In [15]:
test_images = ['Alovera', 'Books', 'Bowling', 'Chess', 'Dolls', 'Snowman', 'Teddy']

for addr in test_images:
    left_image = cv2.imread(f'Stereo_Pairs/{addr}/Image_1.png', cv2.IMREAD_GRAYSCALE)
    right_image = cv2.imread(f'Stereo_Pairs/{addr}/Image_2.png', cv2.IMREAD_GRAYSCALE)

    mv, error, (H, W) = encoder(left=left_image, right=right_image, block_size=16)
    recon = decoder(left=left_image, motion_vectors=mv, residual=error, orig_shape=(H, W), block_size=16)

    psnr = peak_signal_noise_ratio(image1=right_image, image2=recon)

    print(f'Test: {addr} - psnr:{psnr}')

Test: Alovera - psnr:inf
Test: Books - psnr:inf
Test: Bowling - psnr:inf
Test: Chess - psnr:inf
Test: Dolls - psnr:inf
Test: Snowman - psnr:inf
Test: Teddy - psnr:inf


In [10]:
left_image = cv2.imread(f'Stereo_Pairs/Alovera/Image_1.png', cv2.IMREAD_GRAYSCALE)
right_image = cv2.imread(f'Stereo_Pairs/Alovera/Image_2.png', cv2.IMREAD_GRAYSCALE)

mv, error, (H, W) = encoder(left=left_image, right=right_image, block_size=16)
recon = decoder(left=left_image, motion_vectors=mv, residual=error, orig_shape=(H, W), block_size=16)